In [ ]:
from spider_data import get_spider_train,get_spider_val

spider = (get_spider_val(5000))
print()

In [2]:
from train import _get_candidate_labels

spider = get_spider_train(5000)

item = spider[0]

labels = _get_candidate_labels(item["input"][0], item["gold_schema"])
print(labels)  # → an Position von Budget_in_Billions muss 1.0 stehen

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]


In [8]:
from utils import parse_orig_sql

parsed = parse_orig_sql("select * from test")
print()

KeyboardInterrupt: 

In [16]:
from spider_data import get_spider_train, get_spider_val

spider_train = get_spider_train(5000)
spider_val = get_spider_val(5000)

not_found= 0
for item in spider_train:
    if(len(item["gold_schema"]) == 0):
        not_found+=1

print("Nicht gefunden Spider_train:", not_found)
print("Spider_train gesamt:", len(spider_train))

not_found= 0
for item in spider_val:
    if(len(item["gold_schema"]) == 0):
        not_found+=1

print("Nicht gefunden Spider_dev:", not_found)
print("Spider_dev gesamt:", len(spider_val))

Nicht gefunden Spider_train: 251
Spider_train gesamt: 7000
Nicht gefunden Spider_dev: 42
Spider_dev gesamt: 1034


In [22]:
test = "SELECT T1.Name FROM concert AS T1 WHERE T1.Year = 2014 INTERSECT SELECT T1.Name FROM singer AS T1 WHERE T1.Age > 30"
print(parse_orig_sql(test))

{'concert': ['year', 'name'], 'singer': ['age', 'name']}


In [1]:
import re
from spider_data import get_spider_train,get_spider_val

data = get_spider_val(10000)

miss, total = 0, 0
for item in data:
    cand = set()
    for prompt in item["input"]:
        # Regex-Lücken sichtbar machen:
        n_marker = prompt.count("«")
        found = re.findall(r"«\s+(\S+)\s+(\S+)\s*»", prompt)
        if len(found) != n_marker:
            print(f"Regex verliert {n_marker - len(found)} Kandidaten")
        cand |= {(t.lower(), c.lower()) for t, c in found}
    gold = {(t.lower(), c.lower()) for t, cols in item["gold_schema"].items() for c in (cols or [])}
    missing = gold - cand
    miss += len(missing); total += len(gold)
    if missing:
        print("Fehlt:", missing, "| Kandidaten-Beispiel:", sorted(cand)[:3])
print(f"\nStrukturell unerreichbar: {miss}/{total} = {miss/total:.2%}")

D:\Master\Grundprojekt\Grundprojekt_Schema_Linking_Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Strukturell unerreichbar: 0/2865 = 0.00%
